# Feature Engineering & Data Analysis

**Continuing from:** `exploring_cleaning_data.ipynb` — this notebook picks up once the data is *clean*, and turns it into something that can actually answer business questions.

## Why this phase exists

Cleaning makes data **correct** — no missing values, no duplicates, the right dtypes. That's necessary, but it's not the same as data being **useful**. A clean `Order Date` column doesn't tell you whether Mondays sell more than Fridays; a clean `Sales`/`Profit` pair doesn't tell you which customers are actually worth the most. Feature engineering is the step where we *derive* new columns that carry that signal explicitly, and data analysis is the step where we use those columns (plus the originals) to actually answer questions.

Skipping straight from cleaning to charts is a common shortcut — and it produces charts that answer questions nobody asked, because the underlying features were never designed around a question in the first place. This notebook keeps the two steps in order: **know the question → build the feature that answers it → analyze it.**

## The Generic Workflow

This is the same repeatable process regardless of the dataset — the checklist we'll apply to the Superstore data in Part 2 below.

0. **Setup & Recap** — load the cleaned data, confirm it survived the handoff intact.
1. **Define the Analysis Questions** — write down what we're trying to answer *before* building features, so every feature has a reason to exist.
2. **Feature Engineering — Derived Columns** — transform existing columns (dates, numeric ratios, categorical cleanup) into new per-row signal.
3. **Feature Engineering — Aggregated Features** — roll row-level data up into entity-level features (per customer, per product, etc.).
4. **Feature Validation** — sanity-check every new feature immediately: nulls introduced by the transform, implausible ranges, divide-by-zero.
5. **Exploratory Data Analysis** — univariate → bivariate → multivariate, in that order.
6. **Answering the Questions** — go back to Step 1 and directly resolve each question with a specific table or aggregation.
7. **Insight Synthesis** — summarize findings in plain language, including caveats inherited from cleaning decisions.
8. **Handoff Prep** — name and preserve the tables/features the next stage (visualization) will need.

## Part 2 — Applying the Workflow to the Superstore Dataset

From here on, every section is Part 2: the same nine steps, made concrete for `Sample-Superstore2019.csv`.

> **Note on notebooks and kernels:** this notebook does **not** share memory with `exploring_cleaning_data.ipynb` — each `.ipynb` file runs its own kernel. Step 0 below re-loads the raw CSV and re-applies the cleaning decisions already validated there, rather than assuming `df` already exists. This keeps the notebook runnable on its own.

### Step 0 — Setup & Recap

We re-establish the clean baseline before building anything new. Nothing new is decided here, we're just reproducing what the cleaning notebook already validated:

- Drop `Unnamed: 0` and `Row ID` (load artifacts, not data)
- Convert `Order Date` / `Ship Date` to real `datetime`
- Fill the 11 missing `Postal Code` values (all Burlington, VT) with `05401`
- Convert low-cardinality text columns to `category`
- Drop the single true exact-duplicate row

One deliberate omission: the 7 `Order ID` + `Product ID` pairs that share a combination but have different `Quantity`/`Sales` are **not** collapsed. Investigation in the cleaning notebook showed each pair has matching unit price and unit profit — they're legitimate separate order lines, not data-entry errors, and dropping either row would silently discard real revenue.

In [1]:
# Importing the proper and required data packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow

In [2]:
# Reading the clean data and make a copy of the original file to be the single source of truth
df = pd.read_parquet("Sample_Superstore_2019_Clean.parquet", engine="pyarrow")
df.shape

(9993, 20)

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
# Creating the copy of the original DataFrame
df_clean = df.copy()
df_clean.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 9993 entries, 0 to 9992
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order ID        9993 non-null   str           
 1   Order Date      9993 non-null   datetime64[us]
 2   Ship Date       9993 non-null   datetime64[us]
 3   Ship Mode       9993 non-null   category      
 4   Customer ID     9993 non-null   str           
 5   Customer Name   9993 non-null   str           
 6   Segment         9993 non-null   category      
 7   Country/Region  9993 non-null   category      
 8   City            9993 non-null   str           
 9   State           9993 non-null   category      
 10  Postal Code     9993 non-null   str           
 11  Region          9993 non-null   category      
 12  Product ID      9993 non-null   str           
 13  Category        9993 non-null   category      
 14  Sub-Category    9993 non-null   category      
 15  Product Name   

In [6]:
df_clean.isnull().sum()

Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country/Region    0
City              0
State             0
Postal Code       0
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
Quantity          0
Discount          0
Profit            0
dtype: int64

### Step 1 — Define the Analysis Questions

These are the questions the rest of this notebook is built to answer. Every feature engineered in Steps 2–3 exists to serve at least one of these:

1. Which `State` generates the most total `Profit`, and which generates the biggest total loss?
2. Is there a relationship between `Discount` and `Profit`?
3. Does `Ship Mode` relate to profitability or shipping time?
4. Is there seasonality in `Sales` — by month, or by day of week?
5. Who are the most valuable customers, and how would we segment them?

Step 6 will come back to this exact list and answer each one directly.

### Step 2 — Feature Engineering: Derived Columns

Two families of per-row features, built directly from existing columns:

- **Date-based**, from `Order Date` / `Ship Date`: how long shipping took, and what calendar pattern the order falls into (weekday, month, quarter, year, weekend flag) — needed to test the seasonality question (Q4).
- **Numeric transforms**, from `Sales` / `Quantity` / `Discount`: a per-unit price, a profit-margin ratio, and a discount bucket — needed to test the discount question (Q2).

In [7]:
df_clean[["Order Date", "Ship Date"]]

,Order Date,Ship Date
0,2018-11-08,2018-11-11
1,2018-11-08,2018-11-11
2,2018-06-12,2018-06-16
3,2017-10-11,2017-10-18
4,2017-10-11,2017-10-18
...,...,...
9988,2016-01-21,2016-01-23
9989,2019-02-26,2019-03-03
9990,2019-02-26,2019-03-03
9991,2019-02-26,2019-03-03


In [21]:
df_clean["Shipping Days"] = (df_clean["Ship Date"] - df_clean["Order Date"]).dt.days

In [23]:
df_clean["Shipping Days"].mean()

np.float64(3.9580706494546183)

In [24]:
df_clean["Order Date"]

0      2018-11-08
1      2018-11-08
2      2018-06-12
3      2017-10-11
4      2017-10-11
          ...    
9988   2016-01-21
9989   2019-02-26
9990   2019-02-26
9991   2019-02-26
9992   2019-05-04
Name: Order Date, Length: 9993, dtype: datetime64[us]

In [25]:
df_clean["Order Date"].dt.day_name()

0        Thursday
1        Thursday
2         Tuesday
3       Wednesday
4       Wednesday
          ...    
9988     Thursday
9989      Tuesday
9990      Tuesday
9991      Tuesday
9992     Saturday
Name: Order Date, Length: 9993, dtype: str

In [26]:
df_clean["Shipping Days"] = (df_clean["Ship Date"] - df_clean["Order Date"]).dt.days
df_clean.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State,...,Quantity,Discount,Profit,Shipping Datys,Order Weekday,Order Month,Order Quarter,Order Year,Is Weekend,Shipping Days
0,Ca-2018-152156,2018-11-08,2018-11-11,Second Class,Cg-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,2,0.00,41.9136,3,Thursday,11,4,2018,False,3
1,Ca-2018-152156,2018-11-08,2018-11-11,Second Class,Cg-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,3,0.00,219.5820,3,Thursday,11,4,2018,False,3
2,Ca-2018-138688,2018-06-12,2018-06-16,Second Class,Dv-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,...,2,0.00,6.8714,4,Tuesday,6,2,2018,False,4
3,Us-2017-108966,2017-10-11,2017-10-18,Standard Class,So-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,5,0.45,-383.0310,7,Wednesday,10,4,2017,False,7
4,Us-2017-108966,2017-10-11,2017-10-18,Standard Class,So-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,2,0.20,2.5164,7,Wednesday,10,4,2017,False,7


In [27]:

df_clean["Order Weekday"] = df_clean["Order Date"].dt.day_name()
df_clean["Order Month"] = df_clean["Order Date"].dt.month
df_clean["Order Quarter"] = df_clean["Order Date"].dt.quarter
df_clean["Order Year"] = df_clean["Order Date"].dt.year

In [28]:
df_clean.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State,...,Quantity,Discount,Profit,Shipping Datys,Order Weekday,Order Month,Order Quarter,Order Year,Is Weekend,Shipping Days
0,Ca-2018-152156,2018-11-08,2018-11-11,Second Class,Cg-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,2,0.00,41.9136,3,Thursday,11,4,2018,False,3
1,Ca-2018-152156,2018-11-08,2018-11-11,Second Class,Cg-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,3,0.00,219.5820,3,Thursday,11,4,2018,False,3
2,Ca-2018-138688,2018-06-12,2018-06-16,Second Class,Dv-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,...,2,0.00,6.8714,4,Tuesday,6,2,2018,False,4
3,Us-2017-108966,2017-10-11,2017-10-18,Standard Class,So-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,5,0.45,-383.0310,7,Wednesday,10,4,2017,False,7
4,Us-2017-108966,2017-10-11,2017-10-18,Standard Class,So-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,2,0.20,2.5164,7,Wednesday,10,4,2017,False,7


In [29]:
print(df_clean["Order Date"].dt.day_of_week, df_clean["Order Date"].dt.day_name())

0       3
1       3
2       1
3       2
4       2
       ..
9988    3
9989    1
9990    1
9991    1
9992    5
Name: Order Date, Length: 9993, dtype: int32 0        Thursday
1        Thursday
2         Tuesday
3       Wednesday
4       Wednesday
          ...    
9988     Thursday
9989      Tuesday
9990      Tuesday
9991      Tuesday
9992     Saturday
Name: Order Date, Length: 9993, dtype: str


In [30]:
df_clean["Order Date"].dt.dayofweek

0       3
1       3
2       1
3       2
4       2
       ..
9988    3
9989    1
9990    1
9991    1
9992    5
Name: Order Date, Length: 9993, dtype: int32

In [32]:

df_clean["Is Weekend"] = df_clean["Order Date"].dt.dayofweek >= 5

df_clean[["Order Date", "Ship Date", "Shipping Days", "Order Weekday", "Sales", "Is Weekend"]].head(50)

,Order Date,Ship Date,Shipping Days,Order Weekday,Sales,Is Weekend
0,2018-11-08,2018-11-11,3,Thursday,261.9600,False
1,2018-11-08,2018-11-11,3,Thursday,731.9400,False
2,2018-06-12,2018-06-16,4,Tuesday,14.6200,False
3,2017-10-11,2017-10-18,7,Wednesday,957.5775,False
4,2017-10-11,2017-10-18,7,Wednesday,22.3680,False
5,2016-06-09,2016-06-14,5,Thursday,48.8600,False
6,2016-06-09,2016-06-14,5,Thursday,7.2800,False
7,2016-06-09,2016-06-14,5,Thursday,907.1520,False
8,2016-06-09,2016-06-14,5,Thursday,18.5040,False
9,2016-06-09,2016-06-14,5,Thursday,114.9000,False


In [38]:
weekend_sales_test = df_clean.groupby("Is Weekend").agg(
    total_sales = ("Profit", "mean")
)
weekend_sales_test

,total_sales
Is Weekend,
False,28.974414
True,27.829590


In [ ]:
df_clean["Is Weekend"].value_counts()

In [40]:
# Numeric transforms -- needed for the discount question (Q2)
df_clean["Unit Price"] = df_clean["Sales"] / df_clean["Quantity"]
df_clean["Profit Margin"] = (df_clean["Profit"] / df_clean["Sales"])*100
df_clean[["Product ID", "Sales", "Unit Price", "Profit Margin"]]

,Product ID,Sales,Unit Price,Profit Margin
0,Fur-Bo-10001798,261.9600,130.9800,16.00
1,Fur-Ch-10000454,731.9400,243.9800,30.00
2,Off-La-10000240,14.6200,7.3100,47.00
3,Fur-Ta-10000577,957.5775,191.5155,-40.00
4,Off-St-10000760,22.3680,11.1840,11.25
...,...,...,...,...
9988,Fur-Fu-10001889,25.2480,8.4160,16.25
9989,Fur-Fu-10000747,91.9600,45.9800,17.00
9990,Tec-Ph-10003645,258.5760,129.2880,7.50
9991,Off-Pa-10004041,29.6000,7.4000,45.00


> Keep in mind that you see the 'Unit Price' and the 'Profit Margin' for each product in each selling order. Later on we are going to see the aggregated values for the 'Unit Price' for each product.

In [42]:
df_clean.head(10)

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State,...,Profit,Shipping Datys,Order Weekday,Order Month,Order Quarter,Order Year,Is Weekend,Shipping Days,Unit Price,Profit Margin
0,Ca-2018-152156,2018-11-08,2018-11-11,Second Class,Cg-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,41.9136,3,Thursday,11,4,2018,False,3,130.9800,16.00
1,Ca-2018-152156,2018-11-08,2018-11-11,Second Class,Cg-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,219.5820,3,Thursday,11,4,2018,False,3,243.9800,30.00
2,Ca-2018-138688,2018-06-12,2018-06-16,Second Class,Dv-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,...,6.8714,4,Tuesday,6,2,2018,False,4,7.3100,47.00
3,Us-2017-108966,2017-10-11,2017-10-18,Standard Class,So-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,-383.0310,7,Wednesday,10,4,2017,False,7,191.5155,-40.00
4,Us-2017-108966,2017-10-11,2017-10-18,Standard Class,So-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,2.5164,7,Wednesday,10,4,2017,False,7,11.1840,11.25
5,Ca-2016-115812,2016-06-09,2016-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,...,14.1694,5,Thursday,6,2,2016,False,5,6.9800,29.00
6,Ca-2016-115812,2016-06-09,2016-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,...,1.9656,5,Thursday,6,2,2016,False,5,1.8200,27.00
7,Ca-2016-115812,2016-06-09,2016-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,...,90.7152,5,Thursday,6,2,2016,False,5,151.1920,10.00
8,Ca-2016-115812,2016-06-09,2016-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,...,5.7825,5,Thursday,6,2,2016,False,5,6.1680,31.25
9,Ca-2016-115812,2016-06-09,2016-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,...,34.4700,5,Thursday,6,2,2016,False,5,22.9800,30.00


In [44]:
df_clean["Discount Category"] = pd.cut(
    df_clean["Discount"],
    bins = [-0.01, 0, 0.3, 0.6, 0.8, 1.0],
    labels = ["No Discount", "Low Discount", "Medium Discount", "High Discount", "Very High Discount"]
)
df_clean[["Discount", "Discount Category"]].head(50)

,Discount,Discount Category
0,0.00,No Discount
1,0.00,No Discount
2,0.00,No Discount
3,0.45,Medium Discount
4,0.20,Low Discount
5,0.00,No Discount
6,0.00,No Discount
7,0.20,Low Discount
8,0.20,Low Discount
9,0.00,No Discount


In [45]:
df_clean["Discount Category"].value_counts()

Discount Category
No Discount           4798
Low Discount          4029
High Discount          718
Medium Discount        448
Very High Discount       0
Name: count, dtype: int64

In [49]:
df_clean["Discount"].max()

np.float64(0.8)

In [ ]:
df_clean["Discount Category"] = pd.cut(
    df_clean["Discount"],
    bins= [-0.01, 0, 0.20, 0.50, 0.8, 1],
    labels= ["NO DISCOUNT", "LOW DISCOUNT", "MEDIUM DISCOUNT", "HIGH DISCOUNT", "VERY HIGH DISCOUNT"]
)

In [ ]:

# Discount buckets: 0 = no discount, then Low/Medium/High bands.
# Bins chosen from the actual observed range (0.0 to 0.8).
df_clean["Discount Bucket"] = pd.cut(
    df_clean["Discount"],
    bins=[-0.01, 0, 0.2, 0.5, 0.8, 1.0],
    labels=["None", "Low", "Medium", "High", "Very High"],
)


In [ ]:

df_clean[["Sales", "Quantity", "Unit Price", "Profit", "Profit Margin", "Discount", "Discount Category"]].head()

In [ ]:
df_clean.info()

### Step 3 — Feature Engineering: Aggregated Features

Row-level features answer row-level questions. To answer "who are our most valuable customers?" (Q5) we need to roll individual order lines up to one row per entity:

- **Customer-level**: how often they order, how much they spend, how long they've been a customer
- **Product-level**: how much of each product sells, and at what average discount
- **RFM** (Recency / Frequency / Monetary): the classic customer-value framework, built directly from the two aggregates above plus `Order Date`

In [52]:
df_clean["Customer ID"].value_counts()

Customer ID
Wb-21850    37
Ma-17560    34
Jl-15835    34
Pp-18955    34
Jd-15895    32
            ..
Jr-15700     1
Ao-10810     1
Ld-16855     1
Cj-11875     1
Re-19405     1
Name: count, Length: 793, dtype: int64

In [58]:
# Group the customer data.
customer_summary = df_clean.groupby(
    df_clean["Customer ID"]
).agg(
    total_sales = ("Sales", "sum"),
    average_sales = ("Sales", 'mean'),
    order_count = ("Order ID", "nunique"),
    total_profit = ("Profit", "sum"),
    average_discount = ("Discount", "mean")
)
customer_summary

,total_sales,average_sales,order_count,total_profit,average_discount
Customer ID,,,,,
Aa-10315,5563.560,505.778182,5,-362.8825,0.090909
Aa-10375,1056.390,70.426000,9,277.3824,0.080000
Aa-10480,1790.512,149.209333,4,435.8274,0.016667
Aa-10645,5086.935,282.607500,6,857.8033,0.063889
Ab-10015,886.156,147.692667,3,129.3465,0.066667
...,...,...,...,...,...
Xp-21865,2374.658,84.809214,11,621.2300,0.046429
Yc-21895,5454.350,681.793750,5,1305.6290,0.075000
Ys-21880,6720.444,560.037000,8,1778.2923,0.050000


In [ ]:
df_clean["Region"].value_counts()

In [59]:
df_clean.groupby("Region").agg(
    Count_of_Orders = ("Order ID", "nunique"),
    Total_Sales = ("Sales", "sum")
)

,Count_of_Orders,Total_Sales
Region,,
Central,1175,501239.8908
East,1401,678499.8680
South,822,391721.9050
West,1611,725457.8245


In [ ]:
df_clean["Discount Category"].value_counts()

In [60]:
# One row per customer: order count, spend, average order value, tenure
customer_features = df_clean.groupby("Customer ID", observed=True).agg(
    order_count=("Order ID", "nunique"),
    total_spend=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    avg_order_value=("Sales", "mean"),
    first_order=("Order Date", "min"),
    last_order=("Order Date", "max"),
)
customer_features.sort_values("Customer ID", ascending=True).head()

,order_count,total_spend,total_profit,avg_order_value,first_order,last_order
Customer ID,,,,,,
Aa-10315,5,5563.560,-362.8825,505.778182,2016-03-31,2019-06-29
Aa-10375,9,1056.390,277.3824,70.426000,2016-04-21,2019-12-11
Aa-10480,4,1790.512,435.8274,149.209333,2016-05-04,2019-04-15
Aa-10645,6,5086.935,857.8033,282.607500,2016-06-22,2019-11-05
Ab-10015,3,886.156,129.3465,147.692667,2016-02-18,2018-11-10


In [ ]:
customer_features.info()

In [ ]:
# Tenure = span between a customer's first and last order in this dataset
customer_features["Customer Tenure Days"] = (
    customer_features["last_order"] - customer_features["first_order"]
).dt.days

customer_features.sort_values("Customer Tenure Days", ascending=False).head(10)

In [ ]:
# One row per product: units sold, revenue, average discount given
product_features = df_clean.groupby("Product ID", observed=True).agg(
    total_qty_sold=("Quantity", "sum"),
    total_revenue=("Sales", "sum"),
    avg_discount=("Discount", "mean"),
).sort_values("total_revenue", ascending=False)

product_features.head()

In [ ]:
# RFM: Recency (days since last order), Frequency (distinct orders),
# Monetary (total Sales). Recency is measured against the day after the
# last order in the whole dataset, since we have no "today" to measure from.
snapshot_date = df_clean["Order Date"].max() + pd.Timedelta(days=1)
snapshot_date

In [ ]:
rfm = df_clean.groupby("Customer ID", observed=True).agg(
    Recency=("Order Date", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Order ID", "nunique"),
    Monetary=("Sales", "sum"),
)

rfm.sort_values("Monetary", ascending=False).head()

### Step 4 — Feature Validation

New features can silently break things the raw columns never would: a ratio can divide by zero, a date subtraction can go negative if the source dates were wrong, a bin can leave rows unlabeled. Check every new feature once, right after creating it.

In [ ]:
# Divide-by-zero / infinite values in the ratio features
print("Infinite Unit Price values:", np.isinf(df_clean["Unit Price"]).sum())
print("Infinite Profit Margin values:", np.isinf(df_clean["Profit Margin"]).sum())


In [ ]:
# Shipping Days should never be negative -- that would mean a product
# shipped before it was ordered
print("Shipping Days range:", df_clean["Shipping Days"].min(), "to", df_clean["Shipping Days"].max())
print("Negative Shipping Days:", (df_clean["Shipping Days"] < 0).sum())


In [ ]:
# Every row should have landed in exactly one Discount Bucket
print("Unlabeled Discount Bucket rows:", df_clean["Discount Bucket"].isnull().sum())

### Step 5 — Exploratory Data Analysis

Univariate first (what does one feature look like on its own), then bivariate (how does it relate to profit), then multivariate (multiple dimensions at once).

In [ ]:
# Univariate: distribution of the new per-row features
print(df_clean["Discount Bucket"].value_counts())
print()
print(df_clean["Shipping Days"].describe())
print()
print(df_clean["Profit Margin"].describe())

In [ ]:
# Bivariate: Profit against Discount Bucket, Sales against calendar features
print(df_clean.groupby("Discount Bucket", observed=True)["Profit"].mean())
print()
print(df_clean.groupby("Order Month", observed=True)["Sales"].sum())

**Multivariate**: once two dimensions aren't enough, `.groupby()` on multiple columns and `.pivot_table()` spread one categorical column across the columns of the result — closer to the shape a report or chart actually needs.

- `.groupby("column")` splits the DataFrame into groups sharing the same value; chaining an aggregation (`.sum()`, `.mean()`, `.agg()`) collapses each group into one row.
- `.pivot_table()` is the more flexible version — it lets a second categorical column spread across the result's columns.
- `.sort_values()` orders the result so the most/least interesting rows are easy to spot.

In [ ]:
# Total profit and sales per Region, sorted from most to least profitable
region_summary = df_clean.groupby("Region", observed=True)[["Sales", "Profit"]].sum().sort_values("Profit", ascending=False)
region_summary

In [ ]:
# Multiple aggregations at once with .agg()
category_summary = df_clean.groupby("Category", observed=True).agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    orders=("Order ID", "count"),
)
category_summary

In [ ]:
# pivot_table: average Sales by Region (rows) x Category (columns)
pivot = pd.pivot_table(
    df_clean,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="mean",
    observed=True,
)
pivot

### Step 6 — Answering the Questions

Back to the five questions from Step 1 — each one resolved directly with the features built above.

In [ ]:
# Q1: which State generates the most Profit, and which the biggest loss?
state_profit = df_clean.groupby("State", observed=True)["Profit"].sum().sort_values(ascending=False)
print("Highest-profit state:", state_profit.idxmax(), "->", round(state_profit.max(), 2))
print("Biggest-loss state:", state_profit.idxmin(), "->", round(state_profit.min(), 2))

In [ ]:
# Q2: relationship between Discount and Profit, using the bucket built in Step 2
discount_profit = df_clean.groupby("Discount Bucket", observed=True)["Profit"].mean()
discount_profit

In [ ]:
# Q3: does Ship Mode relate to profitability or shipping time?
ship_mode_summary = df_clean.groupby("Ship Mode", observed=True).agg(
    avg_profit=("Profit", "mean"),
    avg_shipping_days=("Shipping Days", "mean"),
)
ship_mode_summary

In [ ]:
# Q4: seasonality in Sales -- by month and by weekday
print(df_clean.groupby("Order Month", observed=True)["Sales"].sum())
print()
print(df_clean.groupby("Order Weekday", observed=True)["Sales"].sum().sort_values(ascending=False))

In [ ]:
# Q5: most valuable customers, using the RFM features from Step 3
rfm.sort_values("Monetary", ascending=False).head()

### Try It Yourself

1. Group by `Sub-Category` and find total `Profit`, sorted ascending (most loss-making first) — which sub-category loses the most money overall?
2. Build a pivot table showing total `Sales` with `Segment` as rows and `Ship Mode` as columns.
3. Using `.groupby()` with `.agg()`, compute both the average and the maximum `Shipping Days` per `Ship Mode`.

In [ ]:
# TODO 1: total Profit per Sub-Category, sorted ascending


# TODO 2: pivot table -- Sales by Segment (rows) x Ship Mode (columns)


# TODO 3: average and max Shipping Days per Ship Mode

### Step 7 — Insight Synthesis

- **Profit by state**: California is the strongest performer (~$76.4K total profit); Texas is the biggest drag (~-$25.7K total loss) — worth checking whether that's driven by discounting practices specific to Texas.
- **Discount vs. Profit**: a clear, monotonic relationship. Average profit per order falls as the discount band rises — roughly +$67 with no discount, +$27 at Low, **-$78 at Medium, -$107 at High**. Discounts above ~20% are, on average, selling at a loss.
- **Ship Mode**: shipping time behaves exactly as expected (Same Day ≈ 0 days, Standard Class ≈ 5 days), but average profit per order is fairly flat across all four modes (~$28–32) — shipping choice doesn't appear to drive profitability on its own.
- **Seasonality**: Sales rise sharply toward year-end (November and December are the two highest months, roughly 3–4x February's total), consistent with holiday retail patterns. By weekday, Monday/Tuesday/Wednesday outsell Friday by close to 2x.
- **Customer value**: spend is concentrated — the top RFM customer alone accounts for ~$25K in lifetime Sales, well above the ~$2.9K average, suggesting a small set of high-value accounts worth treating differently from the broader base.

**Caveat carried from cleaning:** these Profit/Sales totals reflect the decision *not* to collapse the 7 legitimate `Order ID` + `Product ID` duplicate pairs — collapsing them would have understated every total above.

### Step 8 — Handoff Prep for Visualization

The next stage (lectures 7–8) turns these tables into charts. This notebook's job is to make sure nothing needs recomputing there — everything it needs already exists, by name, above.

In [ ]:
# Tables and features the visualization stage will need -- named here so
# nothing has to be recomputed in the next notebook.
handoff_manifest = {
    "df_clean": "row-level cleaned + feature-engineered DataFrame",
    "customer_features": "one row per Customer ID -- spend, order count, tenure",
    "product_features": "one row per Product ID -- units sold, revenue, avg discount",
    "rfm": "one row per Customer ID -- Recency, Frequency, Monetary",
    "state_profit": "total Profit per State",
    "discount_profit": "average Profit per Discount Bucket",
    "ship_mode_summary": "average Profit and Shipping Days per Ship Mode",
    "region_summary": "total Sales and Profit per Region",
    "category_summary": "total Sales, avg Profit, order count per Category",
    "pivot": "average Sales by Region x Category",
}
for name, description in handoff_manifest.items():
    print(f"{name}: {description}")

### Step 9 — Persist Handoff Tables to Disk

Listing the tables above isn't enough on its own -- `visualization_dashboard.ipynb` runs its **own kernel** and has no access to the variables sitting in this one's memory. Every object named in `handoff_manifest` gets written to a `handoff_data/` folder here, and the visualization notebook's Step 0 reads them straight back in.

- **Parquet** for the row-level / per-entity tables (`df_clean`, `customer_features`, `product_features`, `rfm`) -- preserves dtypes (`datetime64`, `category`) exactly like the clean-data handoff from the cleaning notebook.
- **CSV** for the small summary tables (`state_profit`, `discount_profit`, `ship_mode_summary`, `region_summary`, `category_summary`, `pivot`) -- they're already chart-ready and don't carry dtypes worth preserving via Parquet.

In [ ]:
import os

export_dir = "handoff_data"
os.makedirs(export_dir, exist_ok=True)

# Row-level / per-entity tables -- Parquet preserves dtypes
df_clean.to_parquet(f"{export_dir}/df_clean.parquet", engine="pyarrow")
customer_features.to_parquet(f"{export_dir}/customer_features.parquet", engine="pyarrow")
product_features.to_parquet(f"{export_dir}/product_features.parquet", engine="pyarrow")
rfm.to_parquet(f"{export_dir}/rfm.parquet", engine="pyarrow")

# Small, chart-ready summary tables -- CSV is sufficient
state_profit.to_csv(f"{export_dir}/state_profit.csv")
discount_profit.to_csv(f"{export_dir}/discount_profit.csv")
ship_mode_summary.to_csv(f"{export_dir}/ship_mode_summary.csv")
region_summary.to_csv(f"{export_dir}/region_summary.csv")
category_summary.to_csv(f"{export_dir}/category_summary.csv")
pivot.to_csv(f"{export_dir}/pivot_region_category.csv")

print(f"Exported {len(os.listdir(export_dir))} files to '{export_dir}/':")
for filename in sorted(os.listdir(export_dir)):
    print(f"  {filename}")